<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c40-header.png" alt="Nextia Learning · Experiments and Causal Impact: Did Our Change Work?" width="100%">

# Solution: Uplift models

**[Uplift models](https://learning.nextia-ai.com/courses/experiments/m07/uplift-models/)** · Experiments and Causal Impact: Did Our Change Work? · Module 7, lesson 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** train a "who will buy" model and two uplift models (the T-learner and the X-learner) on the randomised email test, and see which customers each one puts at the top of its list.

| | |
|---|---|
| **Time** | About 40 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | The four groups, from [The average hides the people](https://learning.nextia-ai.com/courses/experiments/m07/the-average-hides-the-people/). |
| **Tested** | Python 3.12 with pandas 3.1.1, SciPy 1.15.3, statsmodels 0.15.0, matplotlib 3.11.2 and scikit-learn 1.9.1, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/experiments/m07/uplift-models/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C40/M07-L02-uplift-models/uplift-models-solution.ipynb).

## Setup: install the packages

Run the next cell first. It installs the versions that the course uses: pandas 3.1.1, SciPy 1.15.3 and statsmodels 0.15.0, matplotlib 3.11.2 for the charts, and scikit-learn 1.9.1 for the models. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with these versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 scipy==1.15.3 statsmodels==0.15.0 matplotlib==3.11.2 scikit-learn==1.9.1
import importlib.metadata as _md
for _name in ["pandas", "scipy", "statsmodels", "matplotlib", "scikit-learn"]:
    print(_name, _md.version(_name))

## Setup: get the practice data

Run the next cell. It downloads the simulator's data for this lesson (less than 3 MB) into a folder `larkfield-experiments/data`, and checks each file's checksum. It needs an internet connection and no account.

You should see `Ready: 2 files in data/.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-experiments` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield data is synthetic: the course simulator made it. It has no real people. Each folder has `truth.json`: the true effects that the simulator knows. A real test never has this file.

In [ ]:
# Setup: download the course data into larkfield-experiments/data/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

DATA = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C40/data/"
FILES = {
    "email/customers.csv.gz": "e16d24a15f1260876a6d73117b24604495f4af0abc855978b7e5b2828ecd3d3c",
    "email/truth.json": "80bc73295fefa15bf0567269825b5e71402abdc98180deaa302e3a05396b9fd3"
}
LOCAL = os.environ.get("NX_DATA_LOCAL")  # authors only: a local copy of the labs data
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("larkfield-experiments").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and sha256(path) == expected:
        continue
    if LOCAL:
        shutil.copy(Path(LOCAL) / name, path)
    else:
        download(DATA + name, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-experiments and run again.")
os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in data/.")

## Setup: helpers

The next cell loads pandas as `pd`, NumPy as `np`, SciPy's `stats` and statsmodels' formula interface as `smf`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `truth(name)` reads a scenario's `truth.json`: the simulator's true effects.

Run the cell. It prints nothing.

In [ ]:
# Helpers: checks that do not show the answer, and the simulator's truth.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
pd.set_option("display.precision", 4)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def truth(name):
    """The simulator's true effects for a scenario (a real test never has this)."""
    return json.loads(open(f"data/{name}/truth.json").read())

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The test, split in two

We train on 70% of the test and keep 30% for evaluation. The evaluation group plays no part in the training. The split below is the same as in `xp uplift`.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.linear_model import LogisticRegression

email = pd.read_csv("data/email/customers.csv.gz")
FEATURES = ["new_customer", "tenure_days", "orders_12m", "spend_12m", "days_since_order", "email_open_rate", "mobile"]
email["mobile"] = (email.device == "mobile").astype(int)

rng = np.random.default_rng(0)
is_test = rng.random(len(email)) < 0.3
train, hold = email[~is_test], email[is_test]
X, y, t = train[FEATURES].to_numpy(float), train.ordered.to_numpy(float), (train.variant == "B").to_numpy(int)
Xh = hold[FEATURES].to_numpy(float)
print(f"train: {len(train):,} customers; evaluate: {len(hold):,} customers")

def tree():
    return HistGradientBoostingRegressor(max_iter=150, learning_rate=0.05, max_leaf_nodes=15,
                                         min_samples_leaf=100, random_state=0)

def report(score, name):
    """The B − A difference in the share who ordered, among the top 20% of the evaluation group by this score."""
    top = hold.iloc[np.argsort(-score)[: len(hold) // 5]]
    p = top.groupby("variant")["ordered"].mean()
    print(f"{name:<20} top 20%: effect {p['B'] - p['A']:+.4f}; new customers {top.new_customer.mean():.0%}; "
          f"heavy buyers {(top.orders_12m >= 7).mean():.0%}")
    return float(p["B"] - p["A"])

> **Check.** You should see `train: 42,157 customers; evaluate: 17,843 customers`.

## 2. The "who will buy" model

A normal model: trained on the customers who got the email, it predicts the chance to order when emailed.

> **Predict.** Which customers will it put at the top of its list?

In [ ]:
response = HistGradientBoostingClassifier(max_iter=150, learning_rate=0.05, max_leaf_nodes=15,
                                          min_samples_leaf=100, random_state=0).fit(X[t == 1], y[t == 1])
score_response = response.predict_proba(Xh)[:, 1]
effect_response = report(score_response, "who will buy")

> **Check.** You should see `who will buy         top 20%: effect +0.0597; new customers 58%; heavy buyers 3%`.

## 3. The T-learner

Two models: one for the customers without the email, one for the customers with it. The uplift of a customer is the difference of the two chances.

In [ ]:
model_a = tree().fit(X[t == 0], y[t == 0])
model_b = tree().fit(X[t == 1], y[t == 1])
score_t = model_b.predict(Xh) - model_a.predict(Xh)
effect_t = report(score_t, "T-learner")

> **Check.** You should see `T-learner            top 20%: effect +0.0771; new customers 81%; heavy buyers 0%`.

## 4. The X-learner

The X-learner starts from the same two models. Then it guesses each customer's effect and trains models on those guesses.

- For a customer in group B: result − model A's chance.
- For a customer in group A: model B's chance − result.

In [ ]:
guess_b = y[t == 1] - model_a.predict(X[t == 1])
guess_a = model_b.predict(X[t == 0]) - y[t == 0]
effect_model_b = tree().fit(X[t == 1], guess_b)
effect_model_a = tree().fit(X[t == 0], guess_a)
share_b = LogisticRegression(max_iter=1000).fit(X, t).predict_proba(Xh)[:, 1]
score_x = share_b * effect_model_a.predict(Xh) + (1 - share_b) * effect_model_b.predict(Xh)
effect_x = report(score_x, "X-learner")

> **Check.** You should see `X-learner            top 20%: effect +0.0781; new customers 89%; heavy buyers 0%`.

All three lists have a positive effect in the evaluation group: real effects, measured with the random test. The next lesson compares the whole lists with uplift and Qini curves, and with the simulator's truth.

## 5. Your turn: one number per list

> **Your turn.** Find the share of `score_t`'s top 20% that are customers who ordered at least 7 times in 12 months (heavy buyers). Put it in `t_heavy_share`, rounded to 4 decimals.

In [ ]:
top_t = hold.iloc[np.argsort(-score_t)[: len(hold) // 5]]
t_heavy_share = round(float((top_t.orders_12m >= 7).mean()), 4)
print(t_heavy_share)

In [ ]:
expect('t_heavy_share', t_heavy_share, 0.0039, tolerance=0.0005)

## Next

The [lesson page](https://learning.nextia-ai.com/courses/experiments/m07/uplift-models/) explains why a "who will buy" model targets the wrong people and has the knowledge checks that count toward your certificate. Next, we evaluate the lists and choose whom to email under a budget.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Evaluate and use uplift](https://learning.nextia-ai.com/courses/experiments/m07/evaluate-and-use-uplift/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/experiments/m07/uplift-models/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.